[Reference](https://pub.towardsai.net/vectorless-rag-how-i-built-a-rag-system-without-embeddings-databases-or-vector-similarity-efccf21e42ff)

```
Query → Embedding → Vector DB → Top-k Chunks → LLM → Answer
```

```
# macOS/Linux
python3 -m venv .venv
source .venv/bin/activate
```

```
# Windows
python -m venv .venv
.venv\Scripts\activate
```

```
# Core LLM and Agentic Framework
openai==2.30.0              # OpenAI API client
langgraph==1.1.4            # LangGraph for building state graphs and agents
pydantic==2.12.5            # Data validation
# PDF Processing
PyMuPDF==1.27.2.2           # PDF parsing/manipulation (fitz)
pymupdf4llm                  # Layout-aware PDF to markdown conversion
# Utilities
python-dotenv==1.2.2        # Environment variable management (.env files)
```

# Step 1: Tree Generation

In [1]:
"""
tree.py
-------
Parses a PDF into a hierarchical DocumentTree using PyMuPDF4LLM.

Uses layout-aware PDF parsing without vector embeddings.
Strategy:
1. Extract markdown with layout preservation using PyMuPDF4LLM
2. Parse markdown headers into tree hierarchy
3. Use page_chunks for accurate page boundary detection

Install: pip install PyMuPDF pymupdf4llm
"""

import os
import re
import json
import time
from typing import List, Dict, Optional
from dataclasses import dataclass, field
from pathlib import Path

import fitz  # PyMuPDF
import pymupdf4llm  # Primary parser


# ── Data models ───────────────────────────────────────────────────────────────

@dataclass
class TreeNode:
    """Hierarchical document node"""
    id: str
    title: str
    level: int  # 0=root, 1=chapter, 2=section, 3=subsection
    page_start: int
    page_end: int
    content: str
    children: List['TreeNode'] = field(default_factory=list)
    heading_type: Optional[str] = None  # "numbered", "unnumbered", "page"
    summary: str = ""

    def to_dict(self) -> Dict:
        return {
            "id": self.id,
            "title": self.title,
            "level": self.level,
            "pages": f"{self.page_start}-{self.page_end}",
            "type": self.heading_type,
            "children_count": len(self.children),
            "content_preview": self.content[:200] + "..." if len(self.content) > 200 else self.content
        }


@dataclass
class DocumentTree:
    """Complete document tree with metadata"""
    document_name: str
    root: TreeNode
    total_pages: int
    source_path: str = ""

    def print_tree(self, node: Optional[TreeNode] = None, indent: int = 0):
        """Pretty print tree structure"""
        if node is None:
            node = self.root
            print(f"\n📄 {self.document_name} ({self.total_pages} pages)")

        prefix = "  " * indent
        icon = "📑" if node.level == 0 else "📖" if node.level == 1 else "📄" if node.level == 2 else "📝"
        print(f"{prefix}{icon} [{node.level}] {node.title} (p{node.page_start}-{node.page_end})")

        for child in node.children:
            self.print_tree(child, indent + 1)




class PyMuPDF4LLMTreeBuilder:
    """
    Build hierarchical document trees using PyMuPDF4LLM.
    10x faster than GPU-based methods, no model download required.
    """

    def __init__(self, max_content_length: int = 8000):
        self.max_content_length = max_content_length

        # Heading detection patterns
        self.patterns = {
            'numbered_section': re.compile(r'^(?:\d+\.)+\s+(.+)$'),  # 1. Introduction, 2.3.1 Methods
            'roman_section': re.compile(r'^(?:[IVX]+)\.?\s+(.+)$', re.IGNORECASE),  # I. Introduction, II.3
            'letter_section': re.compile(r'^([A-Z])\.\s+(.+)$'),  # A. Methods, B. Results
            'unnumbered_heading': re.compile(r'^([A-Z][a-zA-Z\s]{3,50})$'),  # Abstract, Conclusion
        }

    def parse_pdf(self, pdf_path: str) -> DocumentTree:
        """
        Parse PDF into hierarchical tree structure.

        Strategy:
        1. Extract markdown with layout preservation using PyMuPDF4LLM
        2. Parse markdown headers into tree hierarchy
        3. Use page_chunks for accurate page boundary detection
        """
        pdf_path = Path(pdf_path)
        print(f"🔍 Parsing {pdf_path.name} with PyMuPDF4LLM...")

        start_time = time.time()

        # Method 1: Get full markdown for structure
        full_md = pymupdf4llm.to_markdown(str(pdf_path))

        # Method 2: Get page chunks for accurate pagination
        page_chunks = pymupdf4llm.to_markdown(
            str(pdf_path),
            page_chunks=True,
            write_images=False,
            embed_images=False
        )

        # Build page index for content lookup
        page_contents = {i+1: chunk["text"] for i, chunk in enumerate(page_chunks)}
        total_pages = len(page_chunks)

        # Parse structure from markdown
        root = self._build_tree_from_markdown(full_md, page_contents, pdf_path.name)

        elapsed = time.time() - start_time
        print(f"✅ Parsed in {elapsed:.2f}s: {total_pages} pages, {self._count_nodes(root)} nodes")

        return DocumentTree(
            document_name=pdf_path.stem,
            root=root,
            total_pages=total_pages,
            source_path=str(pdf_path)
        )

    def _build_tree_from_markdown(
        self,
        markdown: str,
        page_contents: Dict[int, str],
        doc_name: str
    ) -> TreeNode:
        """
        Parse markdown headers into hierarchical tree.
        Handles both numbered and unnumbered headings.
        """
        lines = markdown.split('\n')

        # Create root node
        root = TreeNode(
            id="root",
            title=doc_name,
            level=0,
            page_start=1,
            page_end=max(page_contents.keys()) if page_contents else 1,
            content="",
            heading_type="root"
        )

        # Stack maintains current path: (level, node)
        stack = [(0, root)]
        current_content_lines = []
        current_start_page = 1

        def flush_content():
            """Attach accumulated content to current node"""
            if current_content_lines and stack:
                content = '\n'.join(current_content_lines).strip()
                if content:
                    stack[-1][1].content += "\n\n" + content
                    # Generate summary from first paragraph
                    if not stack[-1][1].summary:
                        first_para = content.replace('#', '').strip()[:300]
                        stack[-1][1].summary = first_para
            current_content_lines.clear()

        i = 0
        while i < len(lines):
            line = lines[i]
            stripped = line.strip()

            # Detect markdown headers
            if stripped.startswith('#'):
                flush_content()

                # Calculate level by counting # characters
                level = len(stripped.split()[0]) if stripped.split() else 0
                title = stripped.lstrip('#').strip()

                # Classify heading type
                heading_type = self._classify_heading(title)

                # Estimate page number based on content position
                # (We'll refine this using page_chunks later)
                page_num = self._estimate_page_number(i, len(lines), max(page_contents.keys()))

                # Create new node
                title_slug = '_'.join(re.findall(r'\w+', title))[:20]
                node_id = f"{title_slug}_{i}"
                new_node = TreeNode(
                    id=node_id,
                    title=title,
                    level=level,
                    page_start=page_num,
                    page_end=page_num,  # Will update later
                    content="",
                    heading_type=heading_type
                )

                # Attach to appropriate parent
                while stack and stack[-1][0] >= level:
                    closed_node = stack.pop()[1]
                    # Update parent's page_end
                    if stack:
                        stack[-1][1].page_end = max(stack[-1][1].page_end, closed_node.page_end)

                if stack:
                    parent = stack[-1][1]
                    parent.children.append(new_node)
                    parent.page_end = max(parent.page_end, page_num)

                stack.append((level, new_node))
                current_start_page = page_num

            else:
                current_content_lines.append(line)

            i += 1

        # Flush final content
        flush_content()

        # Refine page boundaries using page_chunks content matching
        self._refine_page_boundaries(root, page_contents)

        # Distribute content to leaf nodes
        self._distribute_content_to_leaves(root)

        return root

    def _classify_heading(self, title: str) -> str:
        """Classify heading as numbered, roman, letter, or unnumbered."""
        title = title.strip()

        if self.patterns['numbered_section'].match(title):
            return "numbered"
        elif self.patterns['roman_section'].match(title):
            return "roman"
        elif self.patterns['letter_section'].match(title):
            return "letter"
        elif self.patterns['unnumbered_heading'].match(title):
            return "unnumbered"
        else:
            return "unknown"

    def _estimate_page_number(self, line_idx: int, total_lines: int, total_pages: int) -> int:
        """Rough page estimation based on line position."""
        if total_pages == 0:
            return 1
        ratio = line_idx / total_lines if total_lines > 0 else 0
        return min(int(ratio * total_pages) + 1, total_pages)

    def _refine_page_boundaries(self, root: TreeNode, page_contents: Dict[int, str]):
        """
        Refine page boundaries by matching node content to page chunks.
        This corrects the rough estimates from markdown parsing.
        """
        def find_page_for_content(content: str, start_search: int = 1) -> int:
            """Find which page contains this content."""
            content_snippet = content[:100].strip()
            if not content_snippet:
                return start_search

            for page_num, page_text in page_contents.items():
                if page_num >= start_search and content_snippet in page_text:
                    return page_num
            return start_search

        def refine_node(node: TreeNode, parent_start: int = 1):
            # Update start page based on content match
            if node.content:
                matched_page = find_page_for_content(node.content, parent_start)
                node.page_start = matched_page
                node.page_end = matched_page

            # Process children
            prev_end = node.page_start
            for child in node.children:
                refine_node(child, prev_end)
                prev_end = max(prev_end, child.page_end)

            # Update node end to cover all children
            if node.children:
                node.page_end = max(c.page_end for c in node.children)
                node.page_start = min(c.page_start for c in node.children)

        refine_node(root)

    def _distribute_content_to_leaves(self, node: TreeNode):
        """
        Ensure content is stored at appropriate leaf nodes.
        If a node has children, its content becomes a summary.
        """
        if not node.children:
            return

        # Truncate content if node has children (it's a section header)
        if len(node.content) > 500:
            node.summary = node.content[:500] + "..."
            node.content = node.summary

        # Recurse
        for child in node.children:
            self._distribute_content_to_leaves(child)

    def _count_nodes(self, node: TreeNode) -> int:
        """Count total nodes in tree."""
        return 1 + sum(self._count_nodes(c) for c in node.children)


# Public API
def parse_pdf(pdf_path: str) -> DocumentTree:
    """Parse a PDF into a hierarchical document tree using PyMuPDF4LLM."""
    builder = PyMuPDF4LLMTreeBuilder()
    return builder.parse_pdf(pdf_path)

# Step 2: Retrieval (Tree Navigation)

```
Question
   ↓
[Step 1] Analyze Node      ← LLM evaluates relevance and decides next action
   ↓
[Step 2] Route Decision    ← Descend into children, retrieve content, or backtrack
   ↓
[Step 3] Retrieve Content  ← Extract full text from relevant nodes
   ↓
[Step 4] Generate Answer   ← LLM synthesizes final answer with sources
   ↓
Answer + Path + Confidence + Sources
```


In [2]:
"""
retriever.py
------------
Agent-based retrieval that navigates a DocumentTree to answer a query,
with detailed logging of every LLM call and decision.
"""

import json
import re
import time
import logging
from typing import Annotated, Any, Dict, List, Optional, TypedDict
import operator

from langgraph.graph import StateGraph, END
from openai import OpenAI

from tree import TreeNode, DocumentTree

# ── Logger setup ──────────────────────────────────────────────────────────────
#
# Two handlers:
#   console  — INFO and above, human-readable with colour-coded prefixes
#   file     — DEBUG and above, full detail including raw prompts/responses
#
# Usage from outside:
#   import logging
#   logging.getLogger("retriever").setLevel(logging.DEBUG)  # show raw prompts too

logger = logging.getLogger("retriever")
logger.setLevel(logging.DEBUG)
logger.propagate = False   # don't bubble up to root logger

if not logger.handlers:
    # ── Console handler (INFO) ────────────────────────────────────────────
    _ch = logging.StreamHandler()
    _ch.setLevel(logging.INFO)
    _ch.setFormatter(logging.Formatter("%(message)s"))   # raw message only
    logger.addHandler(_ch)

    # ── File handler (DEBUG) ─────────────────────────────────────────────
    _fh = logging.FileHandler("retriever.log", mode="a", encoding="utf-8")
    _fh.setLevel(logging.DEBUG)
    _fh.setFormatter(logging.Formatter(
        "%(asctime)s  %(levelname)-7s  %(message)s",
        datefmt="%H:%M:%S",
    ))
    logger.addHandler(_fh)


# ── Visual helpers ────────────────────────────────────────────────────────────

_DIVIDER   = "─" * 65
_SEPARATOR = "═" * 65

def _box(title: str) -> str:
    return f"\n{_SEPARATOR}\n  {title}\n{_SEPARATOR}"

def _indent(text: str, n: int = 4) -> str:
    pad = " " * n
    return "\n".join(pad + line for line in str(text).splitlines())


# ── State ─────────────────────────────────────────────────────────────────────

class RetrievalState(TypedDict):
    query: str
    current_node: Optional[TreeNode]
    tree: TreeNode
    path_taken: Annotated[List[str], operator.add]
    retrieved_content: Annotated[List[str], operator.add]
    reasoning: str
    confidence: float
    should_descend: bool
    target_child_id: Optional[str]
    depth: int
    final_answer: Optional[str]
    call_log: Annotated[List[dict], operator.add]   # full log of every LLM call


# ── Core LLM caller with logging ──────────────────────────────────────────────

def _call_llm(
    client: OpenAI,
    model: str,
    prompt: str,
    call_type: str,          # "navigate" | "answer"
    call_number: int,
) -> tuple[str, float]:
    """
    Call the LLM and log everything:
      - call number and type
      - full prompt (DEBUG / file only)
      - raw response (DEBUG / file only)
      - latency
    Returns (response_text, elapsed_seconds).
    """
    logger.info(f"\n{_DIVIDER}")
    logger.info(f"  LLM Call #{call_number}  [{call_type.upper()}]")
    logger.info(_DIVIDER)

    # Full prompt goes to file (DEBUG) only — too verbose for console
    logger.debug(f"PROMPT:\n{_indent(prompt)}")

    t0 = time.perf_counter()
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
    )
    elapsed = time.perf_counter() - t0
    raw = response.choices[0].message.content.strip()

    logger.debug(f"RAW RESPONSE:\n{_indent(raw)}")
    logger.info(f"  Model    : {model}")
    logger.info(f"  Latency  : {elapsed:.2f}s")
    logger.info(f"  Tokens   : {response.usage.prompt_tokens} in / "
                f"{response.usage.completion_tokens} out")

    return raw, elapsed


def _strip_fences(text: str) -> str:
    if "```" in text:
        for part in text.split("```"):
            part = part.strip().lstrip("json").strip()
            try:
                json.loads(part)
                return part
            except json.JSONDecodeError:
                continue
    return text


# ── Graph nodes ───────────────────────────────────────────────────────────────

def _make_analyze(client: OpenAI, model: str):
    def analyze_node(state: RetrievalState) -> dict:
        # Handle both TreeNode and DocumentTree objects
        if state["current_node"]:
            node: TreeNode = state["current_node"]
        else:
            # If tree is a DocumentTree, get its root; otherwise use it directly
            tree_obj = state["tree"]
            node = tree_obj.root if hasattr(tree_obj, "root") else tree_obj

        call_num = len(state["call_log"]) + 1
        depth = state["depth"]

        children_info = (
            [{"id": c.id, "title": c.title,
              "summary": getattr(c, "summary", "")[:150]}
             for c in node.children]
            if node.children else []
        )

        # ── Log: entering this node ───────────────────────────────────────
        logger.info(f"\n{'  ' * depth}┌─ Depth {depth} | Node: \"{node.title}\"")
        logger.info(f"{'  ' * depth}│  id={node.id}  pages={node.page_start}-{node.page_end}")
        logger.info(f"{'  ' * depth}│  children={[c.title for c in node.children] or 'none (leaf)'}")

        prompt = f"""You are navigating a research paper tree to answer a query.

Query: "{state['query']}"

Current node:
  id      : {node.id}
  title   : {node.title}
  summary : {getattr(node, 'summary', '')[:300]}
  pages   : {node.page_start}–{node.page_end}
  preview : {node.content[:500] if node.content else 'N/A'}

Children: {json.dumps(children_info, indent=2) if children_info else "None (leaf node)"}

Decide:
1. confidence      : 0–1, how likely does this node (or its children) contain the answer?
2. should_descend  : true only if a specific child is more relevant than this node's content
3. target_child_id : the id of the best child to visit (null if should_descend is false)
4. reasoning       : one sentence explaining your decision

Respond ONLY as valid JSON, no markdown fences:
{{
  "confidence": 0.85,
  "should_descend": true,
  "target_child_id": "1_Introduction_12",
  "reasoning": "The Introduction section directly addresses what Bigtable is."
}}"""

        raw, elapsed = _call_llm(client, model, prompt, "navigate", call_num)
        raw = _strip_fences(raw)

        try:
            decision = json.loads(raw)
        except json.JSONDecodeError:
            logger.warning(f"  [!] JSON parse failed — using fallback decision")
            decision = {
                "confidence": 0.5,
                "should_descend": bool(node.children),
                "target_child_id": node.children[0].id if node.children else None,
                "reasoning": "Fallback: could not parse LLM response",
            }

        conf      = float(decision.get("confidence", 0.5))
        descend   = bool(decision.get("should_descend", False))
        child_id  = decision.get("target_child_id")
        reasoning = decision.get("reasoning", "")

        # ── Log: decision ─────────────────────────────────────────────────
        arrow = "↓ descend" if (descend and node.children) else "→ retrieve"
        logger.info(f"{'  ' * depth}│")
        logger.info(f"{'  ' * depth}│  Decision   : {arrow}")
        logger.info(f"{'  ' * depth}│  Confidence : {conf:.0%}")
        if child_id and descend:
            logger.info(f"{'  ' * depth}│  Next node  : {child_id}")
        logger.info(f"{'  ' * depth}│  Reasoning  : {reasoning}")
        logger.info(f"{'  ' * depth}└─ ({elapsed:.2f}s)")

        entry = {
            "call_number":    call_num,
            "call_type":      "navigate",
            "node_id":        node.id,
            "node_title":     node.title,
            "depth":          depth,
            "confidence":     conf,
            "should_descend": descend,
            "target_child":   child_id,
            "reasoning":      reasoning,
            "latency_s":      round(elapsed, 3),
        }

        return {
            "path_taken":      [node.id],     # LangGraph appends automatically
            "current_node":    node,
            "confidence":      conf,
            "should_descend":  descend,
            "target_child_id": child_id,
            "reasoning":       reasoning,
            "depth":           depth + 1,
            "call_log":        [entry],        # LangGraph appends automatically
        }

    return analyze_node


def _make_descend():
    def descend(state: RetrievalState) -> dict:
        current: TreeNode = state["current_node"]
        target_id: Optional[str] = state.get("target_child_id")
        depth = state["depth"]

        target = next(
            (c for c in current.children if c.id == target_id),
            current.children[0],
        )

        logger.info(f"\n{'  ' * depth}➜  Descending into: \"{target.title}\"")

        return {"current_node": target}

    return descend


def _make_retrieve():
    def retrieve(state: RetrievalState) -> dict:
        node: TreeNode = state["current_node"]
        depth = state["depth"]

        logger.info(f"\n{'  ' * depth}✦  Retrieving content from: \"{node.title}\"")
        logger.info(f"{'  ' * depth}   Pages {node.page_start}–{node.page_end} | "
                    f"{len(node.content)} chars")

        chunk = (
            f"=== **{node.title}** "
            f"(Pages {node.page_start}-{node.page_end}) ===\n"
            f"{node.content}"
        )
        return {"retrieved_content": [chunk]}   # LangGraph appends automatically

    return retrieve


def _make_generate(client: OpenAI, model: str):
    def generate_answer(state: RetrievalState) -> dict:
        call_num  = len(state["call_log"]) + 1
        context   = "\n\n---\n\n".join(state["retrieved_content"])
        sources   = [s.splitlines()[0] for s in state["retrieved_content"]]

        logger.info(f"\n{_DIVIDER}")
        logger.info(f"  Generating answer from {len(state['retrieved_content'])} "
                    f"retrieved section(s):")
        for s in sources:
            logger.info(f"    • {s}")

        prompt = f"""You are an expert on distributed systems and database engineering.
Answer the question using ONLY the retrieved document sections below.
Cite the section title and page range for every claim you make.
If the context is insufficient, say so clearly — do not guess.

Question: {state['query']}

Retrieved sections:
{context}

Answer:"""

        raw, elapsed = _call_llm(client, model, prompt, "answer", call_num)

        entry = {
            "call_number": call_num,
            "call_type":   "answer",
            "sources":     sources,
            "latency_s":   round(elapsed, 3),
        }

        logger.info(f"\n  Answer generated in {elapsed:.2f}s")

        return {
            "final_answer": raw,
            "call_log":     [entry],
        }

    return generate_answer


# ── Routing ───────────────────────────────────────────────────────────────────

MAX_DEPTH = 5

def _route(state: RetrievalState) -> str:
    if state["confidence"] < 0.3:
        logger.info(f"\n  ✗  Low confidence ({state['confidence']:.0%}) — stopping traversal")
        return "end"
    if state["depth"] >= MAX_DEPTH:
        logger.info(f"\n  ⚠  Max depth ({MAX_DEPTH}) reached — retrieving current node")
        return "retrieve"
    if state["should_descend"] and state["current_node"].children:
        return "descend"
    return "retrieve"


# ── Graph assembly ────────────────────────────────────────────────────────────

def _build_graph(client: OpenAI, model: str) -> Any:
    workflow = StateGraph(RetrievalState)

    workflow.add_node("analyze",  _make_analyze(client, model))
    workflow.add_node("descend",  _make_descend())
    workflow.add_node("retrieve", _make_retrieve())
    workflow.add_node("generate", _make_generate(client, model))

    workflow.set_entry_point("analyze")
    workflow.add_conditional_edges(
        "analyze", _route,
        {"descend": "descend", "retrieve": "retrieve", "end": END},
    )
    workflow.add_edge("descend",  "analyze")
    workflow.add_edge("retrieve", "generate")
    workflow.add_edge("generate", END)

    return workflow.compile()


# ── Visualization ─────────────────────────────────────────────────────────────

def generate_workflow_png(output_path: str = "workflow.png") -> str:
    """
    Generate a PNG visualization of the LangGraph workflow structure.

    Args:
        output_path: Path where the PNG will be saved (default: "workflow.png")

    Returns:
        Path to the generated PNG file
    """
    workflow = StateGraph(RetrievalState)

    # Add nodes (dummy functions for structure visualization)
    workflow.add_node("analyze",  lambda state: state)
    workflow.add_node("descend",  lambda state: state)
    workflow.add_node("retrieve", lambda state: state)
    workflow.add_node("generate", lambda state: state)

    workflow.set_entry_point("analyze")
    workflow.add_conditional_edges(
        "analyze", lambda state: "retrieve",
        {"descend": "descend", "retrieve": "retrieve", "end": END},
    )
    workflow.add_edge("descend",  "analyze")
    workflow.add_edge("retrieve", "generate")
    workflow.add_edge("generate", END)

    graph = workflow.compile()
    graph_image = graph.get_graph().draw_mermaid_png()

    with open(output_path, "wb") as f:
        f.write(graph_image)

    logger.info(f"Workflow visualization saved to: {output_path}")
    return output_path


# ── Public API ────────────────────────────────────────────────────────────────

def retrieve(query: str, tree: TreeNode, client: OpenAI, model: str = "gpt-4o-mini") -> Dict:
    """
    Navigate the document tree and answer a query, logging every LLM call.

    Console output (INFO):
        Shows the traversal path, each decision, confidence, and reasoning.

    File output (DEBUG → retriever.log):
        Additionally logs the full prompt and raw LLM response for every call.
    """
    logger.info(_box(f"New Query"))
    logger.info(f"\n  Q: {query}\n")

    graph = _build_graph(client, model)

    t_start = time.perf_counter()

    result = graph.invoke({
        "query":             query,
        "current_node":      None,
        "tree":              tree,
        "path_taken":        [],
        "retrieved_content": [],
        "reasoning":         "",
        "confidence":        0.0,
        "should_descend":    True,
        "target_child_id":   None,
        "depth":             0,
        "final_answer":      None,
        "call_log":          [],
    })

    total_s = time.perf_counter() - t_start
    call_log = result.get("call_log", [])
    nav_calls = sum(1 for c in call_log if c["call_type"] == "navigate")
    ans_calls = sum(1 for c in call_log if c["call_type"] == "answer")

    # ── Final summary ──────────────────────────────────────────────────────
    logger.info(f"\n{_SEPARATOR}")
    logger.info("  RETRIEVAL SUMMARY")
    logger.info(_SEPARATOR)
    logger.info(f"  Total LLM calls : {len(call_log)}  "
                f"({nav_calls} navigate + {ans_calls} answer)")
    logger.info(f"  Path taken      : {' → '.join(result.get('path_taken', []))}")
    logger.info(f"  Total latency   : {total_s:.2f}s")
    logger.info(_SEPARATOR)

    return {
        "answer":     result.get("final_answer") or "No answer generated (low confidence).",
        "path":       result.get("path_taken", []),
        "reasoning":  result.get("reasoning", ""),
        "confidence": result.get("confidence", 0.0),
        "sources":    result.get("retrieved_content", []),
        "call_log":   call_log,
    }

In [3]:
"""
main.py
-------
Vectorless RAG — LangGraph Agent + PDF Tree (no PageIndex)

Flow:
  1. Download Bigtable PDF
  2. Parse PDF → DocumentTree  (one-time, cached to JSON)
  3. For each question: agent traverses the tree → retrieves sections → generates answer

Install:
  pip install PyMuPDF openai langgraph pydantic python-dotenv

.env:
  OPENAI_API_KEY=sk-...
"""

import json
import os
import urllib.request
from pathlib import Path
from dataclasses import asdict

from dotenv import load_dotenv
from openai import OpenAI

from questions import QUESTIONS
from retriever import retrieve, generate_workflow_png
from tree import parse_pdf, TreeNode

load_dotenv()

# ── Config ────────────────────────────────────────────────────────────────────
PDF_URL = (
    "https://static.googleusercontent.com/media/research.google.com"
    "/en//archive/bigtable-osdi06.pdf"
)
PDF_PATH        = Path("bigtable-osdi06.pdf")
TREE_CACHE_PATH = Path("results/document_tree.json")
MODEL           = "gpt-4o-mini"

# ── Init client (one instance, shared across tree.py and retriever.py) ────────
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit(
        "OPENAI_API_KEY not set.\n"
        "Create a .env file with:  OPENAI_API_KEY=sk-..."
    )

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


# ── Step 1: Download PDF ──────────────────────────────────────────────────────
def download_pdf() -> None:
    if PDF_PATH.exists():
        print(f"[✓] PDF already present: {PDF_PATH}")
        return
    print("[↓] Downloading Bigtable paper …")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
    print(f"[✓] Saved → {PDF_PATH}")


# ── Step 2: Build / load tree ─────────────────────────────────────────────────
def dict_to_treenode(data: dict) -> TreeNode:
    """Recursively reconstruct TreeNode from dictionary."""
    children = [
        dict_to_treenode(child) for child in data.get("children", [])
    ]
    data_copy = data.copy()
    data_copy["children"] = children
    return TreeNode(**data_copy)


def get_tree() -> TreeNode:
    """
    Load cached TreeNode from JSON, or build and cache it fresh.
    """
    TREE_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

    if TREE_CACHE_PATH.exists():
        print(f"[✓] Loading cached tree from {TREE_CACHE_PATH}")
        with open(TREE_CACHE_PATH) as f:
            data = json.load(f)
        # Reconstruct TreeNode from cached dict (extract root from DocumentTree)
        tree = dict_to_treenode(data.get("root", data))
        print(f"    {len(tree.children)} sections loaded")
        return tree

    print("[~] Building tree (first run — takes ~10–30 sec with PyMuPDF4LLM) …")
    tree = parse_pdf(str(PDF_PATH))

    with open(TREE_CACHE_PATH, "w") as f:
        json.dump(asdict(tree), f, indent=2, default=str)
    print(f"[✓] Tree cached → {TREE_CACHE_PATH}")
    return tree


# ── Step 3: Ask a question ────────────────────────────────────────────────────
def ask(question: str, tree: TreeNode) -> dict:
    print(f"\n{'─'*70}")
    print(f"  Q: {question}")
    print(f"{'─'*70}")

    # Pass the LLM client to retrieve function
    result = retrieve(question, tree, client)

    print(f"\n  [Reasoning]  {result['reasoning']}")
    print(f"  [Confidence] {result['confidence']:.0%}")
    print(f"  [Path]       {' → '.join(result['path'])}")

    if result["sources"]:
        print(f"\n  [Sources]")
        for src in result["sources"][:2]:
            print(f"    {src.splitlines()[0]}")   # just the header line

    print(f"\n  [Answer]\n{result['answer']}")
    return result


# ── Main ──────────────────────────────────────────────────────────────────────
def main() -> None:
    download_pdf()

    # Load or build the tree
    tree = get_tree()
    # Generate workflow visualization
    TREE_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    workflow_png_path = TREE_CACHE_PATH.parent / "workflow.png"
    generate_workflow_png(output_path=str(workflow_png_path))
    print(f"[✓] Workflow diagram saved → {workflow_png_path}")
    print(f"\n{'═'*70}")
    print("  Vectorless RAG — Google Bigtable (no PageIndex)")
    print(f"{'═'*70}")

    results = []
    for i, question in enumerate(QUESTIONS, 1):
        print(f"\n[{i}/{len(QUESTIONS)}]")
        try:
            result = ask(question, tree)
            results.append({"question": question, "result": result, "ok": True})
        except Exception as e:
            print(f"  [ERROR] {e}")
            results.append({"question": question, "error": str(e), "ok": False})

    ok = sum(r["ok"] for r in results)
    print(f"\n{'═'*70}")
    print(f"  Done: {ok}/{len(results)} questions answered successfully")
    print(f"{'═'*70}\n")


if __name__ == "__main__":
    main()